In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import node2vec
import networkx as nx
import datamapplot
import umap.plot
import torch
from torch_geometric.data import Data, HeteroData
from torch_geometric.utils import to_scipy_sparse_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import HDBSCAN
import umap


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.N

In [3]:
# GOLD = "Data/gold"
GOLD = "Data/data/ACME4/gold/"

# Load the gold train/test splits (one row per process execution).
df_train = pd.read_parquet(f"{GOLD}/train-process_uber_summary.parquet")
df_test  = pd.read_parquet(f"{GOLD}/test-process_uber_summary.parquet")

df_acme = df_train
df_acme.head(5)

,pid_hash,os_family,agent_id,num_agent_id,hostname,os_pid,process_name,num_process_name,args,num_args,...,lolbas_mitre,lolc_class,lolbas_num_rows,mitre_analytic_ids,mitre_information_domains,mitre_subtypes,mitre_analytic_types,mitre_num_rows,bad_user,red_team
0,CF830E1523374BE7F0E5F91F7936443E,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-CWQ,1900,wmic.exe,1,os get version /format:list,1,...,"[T1218, T1564.004]",NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0
1,42DEC9F24F7D08578FB30A75AF68C661,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-DXJ,3748,wmic.exe,1,computersystem get dnshostname /value,1,...,"[T1218, T1564.004]",NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0
2,A4207647384708DBA1C4552112C948F9,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-CWQ,5268,wmic.exe,1,os get version /format:list,1,...,"[T1218, T1564.004]",NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0
3,D7D35ACED1E249E503B9049D2F9599BD,windows,baa6d56b-daab-414f-bfd5-4ffafb247689,1,ACME-HH-CWQ,3056,wmic.exe,1,computersystem get domain /value,1,...,"[T1218, T1564.004]",NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0
4,224B925BA2818C792E07DE5FFD088CCC,windows,f9ac46c8-0959-4bce-82d9-556a971e7f1a,1,ACME-WS-PLU,5160,wmic.exe,1,os get caption /format:list,1,...,"[T1218, T1564.004]",NEUTRAL,1.0,[CAR-2021-05-012],[Analytic],[[Process]],[[TTP]],1.0,None,0



## Torch Geometric Refactor

In [184]:
class GeometricGraphBuilder:
    """
    Create a graph from the passed dataframe and embed it using a graph embedding technique (e.g., node2vec, GCN, etc.).
    Each node in the graph represents a process execution, and edges represent relationships between processes (e.g., parent-child relationship, shared artifact touched, etc.).
    """

    def __init__(self):
        # Initialize class variables, e.g., graph, node/edge feature dictionaries, etc.
        self.sessions = [] # List to store graph sessions (e.g., list of node IDs for each session)

        # Torch Geometric Attributes
        self.node_index = {} # pid_hash -> integer index
        self.node_features = [] # list of float feaure vectors
        self.node_labels = [] # list of int labels

        # File nodes
        self.file_index = {} # filename -> integer index
        self.process_touches_file_edges = [] # process_idx, file_idx

        # Store edges as [src, dst] pairs per type
        self.parent_child_edges = []
        self.rare_artifact_shared_edges = []
        self.same_user_time_window_edges = []

        self.graph = nx.Graph() # Initialize an empty graph using a graph library (e.g., NetworkX, DGL, PyTorch Geometric, etc.)
        self.graph.graph['sessions'] = self.sessions # Attach sessions to the graph object
        
        # Store Torch Geometric Data
        self.data = None

        # Hard coded edge rules for now, can alter values can also do meta param-tuning while training to find best balance of edge weights
        # Maybe implement a meta-tuner inside the training module that automatically finds the best params or let the fit/trainer be able to tune it
        self.EDGE_RULES = {
            "parent_child": {
                "weight": 1.0,
                "reason": "Child Process directly references parent_pid_hash",
            },
            "rare_artifact_shared": {
                "weight": 0.8,
                "reason": "Processes touched the same rare file artifact",
            },
            "same_user_time_window": {
                "weight": 0.5,
                "reason": "Same user and host within specified time window",
            },
        }

    def build_graph(self, df, llm_edge_type_extraction=False):
        """Build the graph from the passed dataframe."""
        print(f"Building graph...")
        
        # Add edges and nodes
        self.add_process_node(df)
        self.add_parent_child_edge(df)
        self.add_rare_artifact_touched_edge(df)
        self.add_same_user_edge(df)

        # Convert to torch geometric representation and extract sessions
        self.data = self._to_pyg()
        self.extract_sessions(self.data)
        
        return self.data

        # User can optionally enable LLM-assisted edge type extraction to identify 
        # additional edge types for graph construction.
        if llm_edge_type_extraction:
            self.add_llm_assisted_edge_type_extraction(df)

        # Extract sesions after graph construction
        self.extract_sessions(self.data)

        return self.graph
    
    def _to_pyg(self):
        """Convert collected lists into a PyTorch Geometric Data object with heterogeneous edge types."""

        data = HeteroData()

        data["process"].x = torch.tensor(self.node_features, dtype=torch.float)
        data["process"].y = torch.tensor(self.node_labels, dtype=torch.long)

        # File nodes have no features - use a dummy 1D feature
        num_files = len(self.file_index)
        data['file'].x = torch.ones((num_files, 1), dtype=torch.float)

        if self.parent_child_edges:
            ei = torch.tensor(self.parent_child_edges, dtype=torch.long).t().contiguous()
            data["process", "parent_child", "process"].edge_index = ei

        if self.same_user_time_window_edges:
            ei = torch.tensor(self.same_user_time_window_edges, dtype=torch.long).t().contiguous()
            data["process", "same_user_time_window", "process"].edge_index = ei

        if self.process_touches_file_edges:
            ei = torch.tensor(self.process_touches_file_edges, dtype=torch.long).t().contiguous()
            data["process", "touches", "file"].edge_index = ei
             # Add reverse so connected_components can traverse both ways
            data["file", "touched_by", "process"].edge_index = ei.flip(0)
        
        return data

    def add_llm_assisted_edge_type_extraction(self, df, description=""):
        """Use LLM-assisted edge type extraction to identify important features for graph construction."""
        print(f"Using LLM-assisted edge type extraction... following this description: {description}")
        pass

    def add_process_node(self, df):
        """Add a node for each process execution in the passed dataframe. (Data)"""
        print(f"Adding process nodes...")
        for idx, row in df.iterrows():
            pid = row["pid_hash"]
            self.node_index[pid] = len(self.node_index) # assign integer index

            # Could expand features or nodes to process, file, ip, domain, user, session
            
            self.node_features.append([
                float(row["duration_seconds"]) if pd.notna(row["duration_seconds"]) else 0.0,
                float(row["num_uniq_file_hash"]) if pd.notna(row["num_uniq_file_hash"]) else 0.0,
                float(row["net_total_events"]) if pd.notna(row["net_total_events"]) else 0.0,
                float(row["conn_id_count"]) if pd.notna(row["conn_id_count"]) else 0.0,
                float(row["reg_totals"]) if pd.notna(row["reg_totals"]) else 0.0,
            ])

            self.node_labels.append(int(row['red_team']) if pd.notna(row['red_team']) else 0)

    def add_parent_child_edge(self, df):
        """This edge type connects a parent process to its child process."""
        print(f"Adding parent-child edges...")
        for idx, row in df.iterrows():
            if pd.notna(row["parent_pid_hash"]):
                src = row["parent_pid_hash"]
                dst = row["pid_hash"]

                if src in self.node_index and dst in self.node_index:
                    src_idx = self.node_index[src]
                    dst_idx = self.node_index[dst]
                    self.parent_child_edges.append((src_idx, dst_idx))

    def add_rare_artifact_touched_edge(self, df):
        """Bipartate: Connect process -> file nodes for touching the same rare artifact."""
        print("Adding rare artifact edges...")
        
        # Step 1: Calculate file frequency across ALL rows
        file_counts = df['filename'].value_counts()
        
        # rare = appears in 2–10 processes (not a common file, not a hub)
        MIN_DEGREE = 2
        # MAX_DEGREE = 10  # ← god-component guard
        MAX_DEGREE = 15  # ← god-component guard
        rare_files = file_counts[
            (file_counts >= MIN_DEGREE) & (file_counts <= MAX_DEGREE)
        ].index

        for idx, row in df.iterrows():
            fname = row["filename"]
            if fname not in rare_files:
                continue

            pid = row["pid_hash"]
            if pid not in self.node_index:
                continue

            # Register file node if new
            if fname not in self.file_index:
                self.file_index[fname] = len(self.file_index) # assign integer index

            self.process_touches_file_edges.append((self.node_index[pid], self.file_index[fname]))
        
    def add_same_user_edge(self, df):
        """Connect processes executed by the same user within a defined time window."""
        print("Adding same user edges...")
        time_window = pd.Timedelta(minutes=5)
        # time_window = pd.Timedelta(hours=1)

        for (user, host), group in df.groupby(["user_name", "hostname"]):
            if pd.isna(user) or pd.isna(host):
                continue

            group = group.sort_values("process_started")
            pid_hashes = group["pid_hash"].tolist()

            for i in range(len(pid_hashes)):
                for j in range(i+1, len(pid_hashes)):
                    time_diff = group.iloc[j]["process_started"] - group.iloc[i]["process_started"]
                    if time_diff <= time_window:
                        if pid_hashes[i] in self.node_index and pid_hashes[j] in self.node_index:
                            self.same_user_time_window_edges.append((self.node_index[pid_hashes[i]], self.node_index[pid_hashes[j]]))
                    else:
                        break


    def embed_graph(self):
        pass
        """Embed the graph using a graph embedding technique (e.g., node2vec, GCN, etc.)."""
        print(f"Embedding {len(self.sessions)} sessions...")

        session_embeddings = []
        session_labels = []

        for session in self.sessions:
            # Numerical features (aggregate with mean)
            node_features = []

            # Categorical Features (aggregate with counts)
            process_names = []
            hostnames = []
            usernames = []

            for node_id in session:
                node = self.graph.nodes[node_id]
                features = [
                    node.get("duration_seconds", 0),
                    node.get("num_file_hash", 0),
                    node.get("net_total_events", 0),
                    node.get("conn_id_count", 0),
                    node.get("reg_totals", 0)
                ]
                node_features.append(features)

                # Collect categorical features
                process_names.append(node.get("process_name", "N/A"))
                hostnames.append(node.get("hostname", "N/A"))
                usernames.append(node.get("user_name", "N/A"))

            # Aggregate numerical features (mean pooling)
            numerical_mean = np.mean(node_features, axis=0)
            numerical_std = np.std(node_features, axis=0)
            numerical_embed = np.concatenate([numerical_mean, numerical_std]) # Combine mean and std

            # Aggregate categorical features (count of unique values)
            categorical_embed = [
                len(set(process_names)), # Number of unique process names in this session
                len(set(hostnames)),     # Number of unique hostnames in this session
                len(set(usernames)),     # Number of unique usernames in this session
                len(session)             # Number of nodes in this session
            ]

            # Combine both
            session_embed = np.concatenate([numerical_embed, categorical_embed])
            session_embeddings.append(session_embed)

            # Label: any red_team occurance means malicious session
            has_malicious = any(self.graph.nodes[n].get("labels") == 1 for n in session)
            session_labels.append(has_malicious)

        return np.array(session_embeddings), np.array(session_labels)
    
    
    def extract_sessions(self, data):
        
        all_edges = torch.cat([
            store.edge_index for store in data.edge_stores if hasattr(store, "edge_index")
        ], dim=1)

        num_nodes = data['process'].num_nodes
        adj = to_scipy_sparse_matrix(all_edges, num_nodes=num_nodes)
        n_components, labels = connected_components(adj, directed=False)

        data['process'].session_id = torch.tensor(labels, dtype=torch.long)
        self.sessions = [(labels == s_id).nonzero()[0].tolist() for s_id in range(n_components)]

        print(f"Extracted {n_components} sessions.")

    def __str__(self):
        desc = f"""
        {self.data}
        \nNodes: {self.data['process'].num_nodes}
        Edges: {self.data['process', 'parent_child', 'process'].edge_index.shape[1] if ('process', 'parent_child', 'process') in self.data else "Empty"}
        Node features: {self.data['process'].x.shape}
        Red team nodes: {self.data['process'].y.sum().item()}
        Sessions: {len(self.sessions)}
        """

        return desc

    def print_node_information(self, idx=None):
        if not self.node_index.items():
            print("No nodes available.")
            return
        
        if idx is not None:
            if idx in self.node_index.values():
                node_data, node_id = next((data, nid) for data, nid in self.node_index.items() if nid == idx)
                print(f"Node {node_id} features: {self.node_features[node_id] if node_id < len(self.node_features) else 'N/A'}")
                print(f"Node {node_id} label: {self.node_labels[node_id] if node_id < len(self.node_labels) else 'N/A'}")
                print(f"Node {node_id} session: {self.sessions[node_id] if node_id < len(self.sessions) else 'N/A'}")
            else:
                print(f"Node {idx} not found.")
        else:
            for node_data, node_id in self.node_index.items():
                print(f"Node {node_id} features: {self.node_features[node_id] if node_id < len(self.node_features) else 'N/A'}")
                print(f"Node {node_id} label: {self.node_labels[node_id] if node_id < len(self.node_labels) else 'N/A'}")
                print(f"Node {node_id} session: {self.sessions[node_id] if node_id < len(self.sessions) else 'N/A'}")

    def print_edge_information(self, edge_type=None):
        if self.data is None:
            print("No graph data available.")
            return None

        if edge_type is not None:
            if edge_type in self.data.edge_types:
                edge_index = self.data[edge_type].edge_index
                info = {
                    "edge_type": edge_type,
                    "edge_index": edge_index,
                    "count": edge_index.shape[1],
                }
                print(f"{edge_type[1]}: {info['count']} edges")
                return info
            print(f"Edge type {edge_type} does not exist.")
            return None

        edge_types = [
            ("process", "parent_child", "process"),
            ("process", "same_user_time_window", "process"),
            ("process", "touches", "file"),
            ("file", "touched_by", "process"),
        ]

        for edge_type in edge_types:
            if edge_type in self.data.edge_types:
                count = self.data[edge_type].edge_index.shape[1]
                print(f"Edge Type: {edge_type} - {edge_type[1]}: {count} edges")
            else:
                print(f"{edge_type[1]}: 0 edges")

In [196]:
builder = GeometricGraphBuilder()

N = 100000
data = builder.build_graph(df_acme.tail(N), llm_edge_type_extraction=True)
node_labels = torch.tensor(builder.node_labels, dtype=torch.long)

Building graph...
Adding process nodes...
Adding parent-child edges...
Adding rare artifact edges...
Adding same user edges...
Extracted 96979 sessions.
